# Data Inspection

This notebook inspects the raw datasets for the Recovery Access Gap Index project and creates the first cleaned municipality and overdose burden outputs.

## Project Setup


In [ ]:
from pathlib import Path

# Notebook is inside /notebooks, so parent folder is the project root
BASE_DIR = Path.cwd().parent
DATA_RAW = BASE_DIR / "data" / "raw"

print("Base directory:", BASE_DIR)
print("Raw data directory exists:", DATA_RAW.exists())

for folder in DATA_RAW.iterdir():
    print(folder.name)


## Inspect MassGIS municipality boundaries


In [ ]:
import geopandas as gpd

massgis_dir = DATA_RAW / "massgis_municipalities"

for shp in massgis_dir.rglob("*.shp"):
    print(shp.name)


In [ ]:
poly_path = massgis_dir / "TOWNSSURVEY_POLY.shp"

towns_poly = gpd.read_file(poly_path)

print("Rows:", len(towns_poly))
print("Unique towns:", towns_poly["TOWN"].nunique())
print("Columns:")
print(towns_poly.columns)

towns_poly.head()


In [ ]:
towns_muni = towns_poly.dissolve(
    by=["TOWN", "TOWN_ID", "COUNTY"],
    as_index=False
)

print("Rows after dissolve:", len(towns_muni))

towns_muni[["TOWN", "TOWN_ID", "COUNTY", "AREA_SQMI"]].head()


In [ ]:
towns_muni.plot()


In [ ]:
processed_dir = BASE_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "ma_municipalities_clean.geojson"

towns_muni.to_file(output_path, driver="GeoJSON")

print("Saved to:", output_path)
print(output_path.exists())


## Inspect Overdose Deaths


In [ ]:
from docx import Document

overdose_dir = DATA_RAW / "overdose_deaths"

print("Files in overdose deaths folder:")
for file in overdose_dir.iterdir():
    print(file.name)

In [ ]:
overdose_file = overdose_dir / "Opioid-related Overdose deaths by City or Town - June 2024.docx"

doc = Document(overdose_file)

print("Number of tables:", len(doc.tables))

for i, table in enumerate(doc.tables):
    print(f"\nTable {i}")
    print("Rows:", len(table.rows))
    print("Columns:", len(table.columns))
    
    # Print first 3 rows from each table
    for row in table.rows[:3]:
        cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
        print(cells)

In [ ]:
import pandas as pd

residence_tables = []

for i, table in enumerate(doc.tables):
    first_cell = table.rows[0].cells[0].text.strip()
    
    if "City/Town of Residence" in first_cell:
        rows = []
        for row in table.rows[1:]:  # skip first repeated header row
            cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
            rows.append(cells)
        
        df = pd.DataFrame(rows)
        df.columns = df.iloc[0]   # use second header row as columns
        df = df.iloc[1:]          # remove header row from data
        
        df["source_table"] = i
        residence_tables.append(df)

overdose_residence = pd.concat(residence_tables, ignore_index=True)

print("Rows:", len(overdose_residence))
print("Columns:")
print(overdose_residence.columns)

overdose_residence.head()

In [ ]:
year_cols = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"]

overdose_clean = overdose_residence.copy()

# Convert year columns to numeric
for col in year_cols:
    overdose_clean[col] = pd.to_numeric(overdose_clean[col], errors="coerce")

# Create project metric: 2021–2023 average annual overdose deaths
overdose_clean["avg_deaths_2021_2023"] = overdose_clean[["2021", "2022", "2023"]].mean(axis=1)

# Standardized town name for joining later
overdose_clean["town_join"] = overdose_clean["City/Town of Residence"].str.upper().str.strip()

overdose_clean[
    ["City/Town of Residence", "2021", "2022", "2023", "avg_deaths_2021_2023", "town_join"]
].head(20)

In [ ]:
overdose_clean.sort_values("avg_deaths_2021_2023", ascending=False)[
    ["City/Town of Residence", "2021", "2022", "2023", "avg_deaths_2021_2023"]
].head(20)

In [ ]:
output_path = processed_dir / "overdose_deaths_by_town_clean.csv"

overdose_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(output_path.exists())

## Inspect EMS Incidents

In [ ]:
ems_dir = DATA_RAW / "ems_incidents"

print("Files in EMS folder:")
for file in ems_dir.iterdir():
    print(file.name)

In [ ]:
ems_file = ems_dir / "Emergency Medical Services Data - June 2024.docx"

ems_doc = Document(ems_file)

print("Number of tables:", len(ems_doc.tables))

for i, table in enumerate(ems_doc.tables):
    print(f"\nTable {i}")
    print("Rows:", len(table.rows))
    print("Columns:", len(table.columns))
    
    for row in table.rows[:3]:
        cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
        print(cells)

In [ ]:
ems_tables = []

for i, table in enumerate(ems_doc.tables):
    first_cell = table.rows[0].cells[0].text.strip()
    
    if "All Suspected Opioid-Related Incidents by Town" in first_cell:
        rows = []
        
        # Skip title row and year row, use the third row as header
        for row in table.rows[2:]:
            cells = [cell.text.strip().replace("\n", " ") for cell in row.cells]
            rows.append(cells)
        
        df = pd.DataFrame(rows)
        df.columns = [
            "city_town",
            "2022_q1", "2022_q2", "2022_q3", "2022_q4", "2022_total",
            "2023_q1", "2023_q2", "2023_q3", "2023_q4", "2023_total"
        ]
        
        # Remove repeated header rows if any
        df = df[df["city_town"] != "City/Town"]
        
        df["source_table"] = i
        ems_tables.append(df)

ems_incidents = pd.concat(ems_tables, ignore_index=True)

print("Rows:", len(ems_incidents))
print("Columns:")
print(ems_incidents.columns)

ems_incidents.head()

In [ ]:
ems_clean = ems_incidents.copy()

def clean_ems_value(value):
    value = str(value).strip()
    
    if value in ["†", ""]:
        return 0
    
    if value in ["(1-4)", "*"]:
        return None
    
    value = value.replace(",", "").replace("±", "")
    
    try:
        return float(value)
    except ValueError:
        return None

for col in ["2022_total", "2023_total"]:
    ems_clean[col] = ems_clean[col].apply(clean_ems_value)

ems_clean["avg_ems_incidents_2022_2023"] = ems_clean[["2022_total", "2023_total"]].mean(axis=1)
ems_clean["town_join"] = ems_clean["city_town"].str.upper().str.strip()

ems_clean[["city_town", "2022_total", "2023_total", "avg_ems_incidents_2022_2023", "town_join"]].head(20)

In [ ]:
ems_clean.sort_values("avg_ems_incidents_2022_2023", ascending=False)[
    ["city_town", "2022_total", "2023_total", "avg_ems_incidents_2022_2023"]
].head(20)

In [ ]:
output_path = processed_dir / "ems_incidents_by_town_clean.csv"

ems_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(output_path.exists())

## Inspect SAMHSA treatment facilities

In [ ]:
samhsa_dir = DATA_RAW / "samhsa_treatment"

print("Files in SAMHSA folder:")
for file in samhsa_dir.iterdir():
    print(file.name)

In [ ]:
samhsa_csv = samhsa_dir / "samhsa_ma_treatment_facilities.csv"

samhsa = pd.read_csv(samhsa_csv)

print("Rows:", len(samhsa))
print("Columns:")
print(samhsa.columns.tolist())

samhsa.head()

In [ ]:
for i, col in enumerate(samhsa.columns):
    print(i, col)

In [ ]:
samhsa_clean = samhsa.copy()

# Keep core facility/location columns first
core_cols = [
    "name1",
    "name2",
    "street1",
    "street2",
    "city",
    "state",
    "zip",
    "county",
    "phone",
    "website",
    "latitude",
    "longitude",
    "type_facility",
    "sa",
    "dt",
    "mm",
    "dm",
    "bum",
    "db",
    "rpn",
    "bu",
    "nxn",
    "vtrl",
    "meth"
]

# Keep only columns that exist, just in case
core_cols = [col for col in core_cols if col in samhsa_clean.columns]

samhsa_clean = samhsa_clean[core_cols].copy()

# Standardize joining/location fields
samhsa_clean["city_join"] = samhsa_clean["city"].str.upper().str.strip()
samhsa_clean["county_join"] = samhsa_clean["county"].str.upper().str.strip()

# Convert coordinates to numeric
samhsa_clean["latitude"] = pd.to_numeric(samhsa_clean["latitude"], errors="coerce")
samhsa_clean["longitude"] = pd.to_numeric(samhsa_clean["longitude"], errors="coerce")

print("Rows:", len(samhsa_clean))
print("Columns:", samhsa_clean.columns.tolist())
print("Missing latitude:", samhsa_clean["latitude"].isna().sum())
print("Missing longitude:", samhsa_clean["longitude"].isna().sum())

samhsa_clean.head()

In [ ]:
samhsa_clean[["name1", "city", "county", "latitude", "longitude", "type_facility"]].head(20)

In [ ]:
output_path = processed_dir / "samhsa_treatment_facilities_clean.csv"

samhsa_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(output_path.exists())

In [ ]:
print("Facilities:", len(samhsa_clean))
print("Cities represented:", samhsa_clean["city_join"].nunique())
print("Counties represented:", samhsa_clean["county_join"].nunique())

samhsa_clean["city_join"].value_counts().head(20)

## Inspect CDC/ATSDR Social Vulnerability Index

## SVI tract-level file was inspected and saved as `svi_tract_clean.csv`.

In [ ]:
svi_dir = DATA_RAW / "svi"

print("Files in SVI folder:")
for file in svi_dir.iterdir():
    print(file.name)

In [ ]:
svi_csv = svi_dir / "svi_2022_massachusetts_tract.csv"

svi = pd.read_csv(svi_csv)

print("Rows:", len(svi))
print("Columns:", len(svi.columns))

for i, col in enumerate(svi.columns):
    print(i, col)

svi.head()

In [ ]:
svi_key_cols = [
    "ST",
    "STATE",
    "ST_ABBR",
    "COUNTY",
    "FIPS",
    "LOCATION",
    "AREA_SQMI",
    "E_TOTPOP",
    "RPL_THEMES",
    "RPL_THEME1",
    "RPL_THEME2",
    "RPL_THEME3",
    "RPL_THEME4",
    "EP_NOVEH",
    "EPL_NOVEH",
    "EP_POV150",
    "EPL_POV150",
    "EP_UNEMP",
    "EPL_UNEMP",
    "EP_NOHSDP",
    "EPL_NOHSDP"
]

existing_svi_cols = [col for col in svi_key_cols if col in svi.columns]

print("Found key columns:")
for col in existing_svi_cols:
    print(col)

print("\nMissing key columns:")
for col in svi_key_cols:
    if col not in svi.columns:
        print(col)

svi[existing_svi_cols].head()

In [ ]:
svi[existing_svi_cols].describe()

In [ ]:
svi_clean = svi[
    [
        "ST",
        "FIPS",
        "AREA_SQMI",
        "E_TOTPOP",
        "RPL_THEMES",
        "RPL_THEME1",
        "RPL_THEME2",
        "RPL_THEME3",
        "RPL_THEME4",
        "EP_NOVEH",
        "EPL_NOVEH",
        "EP_POV150",
        "EPL_POV150",
        "EP_UNEMP",
        "EPL_UNEMP",
        "EP_NOHSDP",
        "EPL_NOHSDP",
    ]
].copy()

# Convert key numeric columns
numeric_cols = [
    "AREA_SQMI",
    "E_TOTPOP",
    "RPL_THEMES",
    "RPL_THEME1",
    "RPL_THEME2",
    "RPL_THEME3",
    "RPL_THEME4",
    "EP_NOVEH",
    "EPL_NOVEH",
    "EP_POV150",
    "EPL_POV150",
    "EP_UNEMP",
    "EPL_UNEMP",
    "EP_NOHSDP",
    "EPL_NOHSDP",
]

for col in numeric_cols:
    svi_clean[col] = pd.to_numeric(svi_clean[col], errors="coerce")

print("Rows:", len(svi_clean))
print("Missing overall SVI:", svi_clean["RPL_THEMES"].isna().sum())
print("Missing no-vehicle percentile:", svi_clean["EPL_NOVEH"].isna().sum())

svi_clean.head()

In [ ]:
output_path = processed_dir / "svi_tract_clean.csv"

svi_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(output_path.exists())

### SVI inspection status

SVI tract-level data inspected and saved as `data/processed/svi_tract_clean.csv` locally. Outputs were cleared before committing for a cleaner GitHub notebook view.


## Inspect manual peer recovery centers

In [ ]:
manual_dir = DATA_RAW / "manual_sources"

peer_csv = manual_dir / "peer_recovery_centers.csv"

peer_recovery = pd.read_csv(peer_csv)

print("Rows:", len(peer_recovery))
print("Columns:")
print(peer_recovery.columns.tolist())

peer_recovery.head()

In [ ]:
print("Missing names:", peer_recovery["name"].isna().sum())
print("Missing addresses:", peer_recovery["address"].isna().sum())
print("Missing cities:", peer_recovery["city"].isna().sum())
print("Missing ZIPs:", peer_recovery["zip"].isna().sum())

peer_recovery[peer_recovery["zip"].isna() | (peer_recovery["zip"].astype(str).str.strip() == "")]

In [ ]:
peer_recovery_clean = peer_recovery.copy()

peer_recovery_clean["city_join"] = peer_recovery_clean["city"].str.upper().str.strip()
peer_recovery_clean["state"] = peer_recovery_clean["state"].str.upper().str.strip()
peer_recovery_clean["service_type"] = "peer_recovery_center"

output_path = processed_dir / "peer_recovery_centers_clean.csv"

peer_recovery_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print(output_path.exists())

In [ ]:
print("Peer recovery rows:", len(peer_recovery_clean))
print("Saved file exists:", output_path.exists())

peer_recovery_clean[["name", "city", "state", "zip", "service_type"]].head(10)


In [ ]:
peer_recovery_clean["city_join"].value_counts().head(20)

## Inspect Manual Syringe Service Programs

In [ ]:
ssp_csv = DATA_RAW / "manual_sources" / "syringe_service_programs.csv"

ssp = pd.read_csv(ssp_csv, dtype={"zip": "string"})
print("Rows:", len(ssp))

ssp[ssp.duplicated(subset=["name", "address", "city"], keep=False)]

In [ ]:
ssp["city"].str.upper().value_counts()

In [ ]:
ssp[ssp.duplicated(keep=False)]

In [ ]:
ssp = pd.read_csv(ssp_csv, dtype={"zip": "string"})
print("Rows:", len(ssp))
ssp.head()

## Scrape and inspect harm reduction Tableau locator

In [ ]:
harm_scraped_path = DATA_RAW / "manual_sources" / "harm_reduction_programs_scraped_from_tableau.csv"
harm_review_path = DATA_RAW / "manual_sources" / "harm_reduction_scraped_needs_review.csv"

harm_scraped = pd.read_csv(harm_scraped_path, dtype={"zip": "string"})
harm_review = pd.read_csv(harm_review_path, dtype={"zip": "string"})

print("Scraped rows:", len(harm_scraped))
print("Review rows:", len(harm_review))

harm_scraped[harm_scraped["city"].str.upper().eq("MA")][["address", "city", "zip"]].head(30)

In [ ]:
harm_review[["address", "city", "zip"]].head(30)

In [ ]:
import re
import pandas as pd

harm_scraped_path = DATA_RAW / "manual_sources" / "harm_reduction_programs_scraped_from_tableau.csv"
harm_review_path = DATA_RAW / "manual_sources" / "harm_reduction_scraped_needs_review.csv"

harm_scraped = pd.read_csv(harm_scraped_path, dtype={"zip": "string"})
harm_review = pd.read_csv(harm_review_path, dtype={"zip": "string"})

def parse_city_zip_flexible(address):
    address = str(address).strip()

    # Pattern: Concord, MA, 01742
    m = re.search(r",\s*([^,]+),\s*MA,?\s*(\d{5}(?:-\d{4})?)?\s*$", address, re.I)
    if m:
        return m.group(1).strip(), "MA", (m.group(2) or "").strip()

    # Pattern: North Adams MA
    m = re.search(r",\s*([^,]+)\s+MA\.?\s*$", address, re.I)
    if m:
        return m.group(1).strip(), "MA", ""

    # Pattern: Brockton MA, 02301
    m = re.search(r",\s*([^,]+)\s+MA,?\s*(\d{5}(?:-\d{4})?)\s*$", address, re.I)
    if m:
        return m.group(1).strip(), "MA", m.group(2).strip()

    return "", "MA", ""

fixed_review = harm_review.copy()

parsed = fixed_review["address"].apply(parse_city_zip_flexible)
fixed_review["city"] = parsed.apply(lambda x: x[0])
fixed_review["state"] = parsed.apply(lambda x: x[1])
fixed_review["zip"] = parsed.apply(lambda x: x[2])

print("Still missing city after fix:", fixed_review["city"].eq("").sum())
fixed_review[fixed_review["city"].eq("")][["address", "city", "zip"]]

In [ ]:
fixed_review.loc[fixed_review["address"].str.contains("Alternative Living Centers", case=False, na=False), "city"] = "North Adams"
fixed_review.loc[fixed_review["address"].str.contains("Alternative Living Centers", case=False, na=False), "zip"] = ""

In [ ]:
fixed_review[fixed_review["city"].eq("")][["address", "city", "zip"]]

In [ ]:
harm_combined = pd.concat([harm_scraped, fixed_review], ignore_index=True)

harm_combined["city"] = harm_combined["city"].str.strip()
harm_combined.loc[
    harm_combined["city"].str.upper().isin(["INDIAN ORCHARD", "INDIAN ORCHARDS"]),
    "city"
] = "Springfield"

harm_combined["city_join"] = harm_combined["city"].str.upper().str.strip()
harm_combined["state"] = "MA"
harm_combined["service_type"] = "harm_reduction_program"

harm_combined["address_key"] = (
    harm_combined["address"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

harm_combined = harm_combined.drop_duplicates(subset=["address_key"]).drop(columns=["address_key"])

output_path = processed_dir / "harm_reduction_programs_clean.csv"
harm_combined.to_csv(output_path, index=False)

print("Final harm reduction rows:", len(harm_combined))
print("Cities represented:", harm_combined["city_join"].nunique())
print("Saved:", output_path)
print(output_path.exists())

harm_combined["city_join"].value_counts().head(25)

In [ ]:
bad_ma = harm_combined[harm_combined["city_join"].eq("MA")][["address", "city", "zip"]]
bad_ma

In [ ]:
def repair_city_from_address(address):
    address = str(address).strip()

    # Handles: "42 Summer Street, 2nd Floor/Suite 201, Pittsfield, MA 01201"
    m = re.search(r",\s*([^,]+),\s*MA\.?\s*(\d{5}(?:-\d{4})?)?", address, re.I)
    if m:
        return m.group(1).strip(), (m.group(2) or "").strip()

    # Handles: "1985 Main St., Suite G, Springfield, MA 01089"
    m = re.search(r",\s*([^,]+)\s*,\s*MA", address, re.I)
    if m:
        return m.group(1).strip(), ""

    return "", ""

bad_mask = harm_combined["city_join"].eq("MA")

for idx, row in harm_combined[bad_mask].iterrows():
    city, zip_code = repair_city_from_address(row["address"])
    if city:
        harm_combined.at[idx, "city"] = city
        harm_combined.at[idx, "city_join"] = city.upper().strip()
    if zip_code:
        harm_combined.at[idx, "zip"] = zip_code

print("Remaining MA city rows:", harm_combined["city_join"].eq("MA").sum())
harm_combined[harm_combined["city_join"].eq("MA")][["address", "city", "zip"]]

In [ ]:
import re

def better_city_zip_from_address(address):
    address = str(address).strip()

    # Get ZIP if present
    zip_match = re.search(r"\b0\d{4}(?:-\d{4})?\b", address)
    zip_code = zip_match.group(0) if zip_match else ""

    # Remove ZIP temporarily
    no_zip = re.sub(r"\b0\d{4}(?:-\d{4})?\b", "", address).strip()

    # Split by commas
    parts = [p.strip() for p in no_zip.split(",") if p.strip()]

    # Find the part that is MA / Mass / Massachusetts, then city is usually before it
    for i, part in enumerate(parts):
        if re.fullmatch(r"MA\.?|Massachusetts", part, flags=re.I):
            if i > 0:
                return parts[i - 1].strip(), zip_code

    # Handles cases like "Pittsfield MA" or "Springfield M.A."
    m = re.search(r"([A-Za-z .'-]+)\s+M\.?A\.?\s*$", no_zip, re.I)
    if m:
        city = m.group(1).strip()
        # If city still has commas, take last comma-separated piece
        city = city.split(",")[-1].strip()
        return city, zip_code

    return "", zip_code


bad_mask = harm_combined["city_join"].eq("MA")

for idx, row in harm_combined[bad_mask].iterrows():
    city, zip_code = better_city_zip_from_address(row["address"])
    if city:
        harm_combined.at[idx, "city"] = city
        harm_combined.at[idx, "city_join"] = city.upper().strip()
    if zip_code:
        harm_combined.at[idx, "zip"] = zip_code

print("Remaining MA city rows:", harm_combined["city_join"].eq("MA").sum())

harm_combined[harm_combined["city_join"].eq("MA")][["address", "city", "zip"]]

In [ ]:
output_path = processed_dir / "harm_reduction_programs_clean.csv"
harm_combined.to_csv(output_path, index=False)

print("Final harm reduction rows:", len(harm_combined))
print("Cities represented:", harm_combined["city_join"].nunique())
print("Bad MA city rows:", harm_combined["city_join"].eq("MA").sum())
print("Saved:", output_path)
print(output_path.exists())

harm_combined["city_join"].value_counts().head(25)

In [ ]:
from difflib import SequenceMatcher
import pandas as pd
import re

def normalize_address(s):
    s = str(s).upper().strip()
    s = re.sub(r"\bSTREET\b", "ST", s)
    s = re.sub(r"\bST\.\b", "ST", s)
    s = re.sub(r"\bAVENUE\b", "AVE", s)
    s = re.sub(r"\bAVE\.\b", "AVE", s)
    s = re.sub(r"\bROAD\b", "RD", s)
    s = re.sub(r"\bRD\.\b", "RD", s)
    s = re.sub(r"\bSUITE\b", "STE", s)
    s = re.sub(r"\bFLOOR\b", "FL", s)
    s = re.sub(r"\bMAIN STREET\b", "MAIN ST", s)
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def street_number(s):
    m = re.search(r"\b\d{1,5}\b", str(s))
    return m.group(0) if m else ""

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

harm_check = harm_combined.copy()
ssp_check = ssp.copy()
peer_check = peer.copy()

harm_check["addr_norm"] = harm_check["address"].apply(normalize_address)
ssp_check["addr_norm"] = ssp_check["address"].apply(normalize_address)
peer_check["addr_norm"] = peer_check["address"].apply(normalize_address)

harm_check["street_num"] = harm_check["address"].apply(street_number)
ssp_check["street_num"] = ssp_check["address"].apply(street_number)
peer_check["street_num"] = peer_check["address"].apply(street_number)

harm_check["city_norm"] = harm_check["city"].astype(str).str.upper().str.strip()
ssp_check["city_norm"] = ssp_check["city"].astype(str).str.upper().str.strip()
peer_check["city_norm"] = peer_check["city"].astype(str).str.upper().str.strip()

def find_possible_dupes(left, right, right_label, threshold=0.70):
    matches = []

    for _, h in left.iterrows():
        same_city = right[right["city_norm"] == h["city_norm"]]

        for _, r in same_city.iterrows():
            score = similarity(h["addr_norm"], r["addr_norm"])

            same_number = (
                h["street_num"] != ""
                and r["street_num"] != ""
                and h["street_num"] == r["street_num"]
            )

            if score >= threshold or (same_number and score >= 0.55):
                matches.append({
                    "harm_name": h["name"],
                    "harm_address": h["address"],
                    "harm_city": h["city"],
                    f"{right_label}_name": r["name"],
                    f"{right_label}_address": r["address"],
                    f"{right_label}_city": r["city"],
                    "similarity": round(score, 3)
                })

    df = pd.DataFrame(matches)

    if len(df) == 0:
        return pd.DataFrame(columns=[
            "harm_name", "harm_address", "harm_city",
            f"{right_label}_name", f"{right_label}_address", f"{right_label}_city",
            "similarity"
        ])

    return df.sort_values("similarity", ascending=False)

possible_ssp_dupes_df = find_possible_dupes(harm_check, ssp_check, "ssp", threshold=0.70)
possible_peer_dupes_df = find_possible_dupes(harm_check, peer_check, "peer", threshold=0.70)

print("Possible harm vs SSP duplicates:", len(possible_ssp_dupes_df))
print("Possible harm vs peer duplicates:", len(possible_peer_dupes_df))

possible_ssp_dupes_df.head(30)

In [ ]:
possible_ssp_dupes_df[
    ["harm_address", "harm_city", "ssp_name", "ssp_address", "ssp_city", "similarity"]
].head(20)

In [ ]:
possible_peer_dupes_df[
    ["harm_address", "harm_city", "peer_name", "peer_address", "peer_city", "similarity"]
].head(20)

In [ ]:
ssp_overlap_addresses = [
    "40 School Street, Unit 6 ,Greenfield, MA 01301",
    "165 Southbridge Street, Worcester, MA 01608",
    "192 Appleton St, Lowell, MA 01852",
    "17 East Silver Street, Westfield, MA 01085",
    "385 Court Street, Plymouth, MA  02360",
    "33 Commercial Street, Gloucester, MA 01930",
    "1985 Main Street, Springfield, MA 01103",
    "75 Amory Street, Boston, MA, 02130",
    "6 West Main Street, North Adams, MA 01247",
    "100 Water Street, Lawrence, MA 01841",
    "9 Winthrop Street, Taunton, MA 02764",
    "1 Welcome Street, Haverhill, MA 01830",
    "359 Green Street, Cambridge, MA, 02139",
]

In [ ]:
peer_overlap_addresses = [
    "35 Exchange St., Lynn, MA 01901",
    "85 Quincy Ave, Quincy, MA 02169",
    "297 Central St., Gardner Ma. 01440",
    "37 Main Street, North Adams MA",
    "52 Main St, Ware, MA 01082",
]

In [ ]:
overlap_addresses = ssp_overlap_addresses + peer_overlap_addresses

harm_deduped = harm_combined[
    ~harm_combined["address"].isin(overlap_addresses)
].copy()

output_path = processed_dir / "harm_reduction_programs_deduped_clean.csv"
harm_deduped.to_csv(output_path, index=False)

print("Original harm rows:", len(harm_combined))
print("Removed overlaps:", len(harm_combined) - len(harm_deduped))
print("Deduped harm rows:", len(harm_deduped))
print("Saved:", output_path)
print(output_path.exists())

harm_deduped["city_join"].value_counts().head(25)

In [ ]:
print("Harm reduction original rows:", len(harm_combined))
print("Harm reduction deduped rows:", len(harm_deduped))
print("Overlaps removed:", len(harm_combined) - len(harm_deduped))
print("Cities represented:", harm_deduped["city_join"].nunique())

harm_deduped["city_join"].value_counts().head(25)